# Aula 23 — Extração estruturada de texto e busca semântica

Uma equipe de comunicação recebe dezenas de notícias por dia. Ela quer responder perguntas práticas: **quem fez o quê, qual tema apareceu e onde está a evidência?** Uma tabela de fatos ajuda a montar um clipping, localizar pautas e conferir afirmações antes de publicar um relatório. Uma busca por significado ajuda quando a pergunta usa palavras diferentes das manchetes.

Hoje faremos uma demonstração geral e iniciaremos o **Projeto 4 (peso 2)**: um recorte de cobertura de Lula e **Flávio Bolsonaro**. O projeto combina o que você aprendeu nas Aulas 21–23. A base do projeto é pequena e fechada: 40 títulos e resumos publicados por Agência Brasil e CNN Brasil, com links e datas. Analisaremos **esse recorte**, não a cobertura total dos veículos.

## 1. Da notícia à linha de uma tabela

Leia a linha P01. Que campos alguém em uma redação procuraria? Um esquema obriga a dizer quais campos existem e a permitir ausentes. A evidência precisa vir do texto recebido, não da memória do modelo. Na tabela, `foco_amostral` indica por que a matéria entrou na base; não é um rótulo de avaliação positiva ou negativa.

In [ ]:
!pip install dspy sentence-transformers

In [8]:
import pandas as pd
from pathlib import Path
arquivo_local = Path("dados/noticias_lula_flavio_2026.csv")
if not arquivo_local.exists():
    arquivo_local = Path("../dados/noticias_lula_flavio_2026.csv")
url_base = "https://raw.githubusercontent.com/mateuspestana/extracao_analise_2026/main/aulas/23-extracao-busca-semantica/dados/noticias_lula_flavio_2026.csv"
politica = pd.read_csv(arquivo_local if arquivo_local.exists() else url_base)
print(politica.shape)
print(pd.crosstab(politica["veiculo"], politica["foco_amostral"]))
assert len(politica) == 40

(40, 8)
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                10    10
CNN Brasil                    10    10


In [9]:
print(politica.loc[0, ["id", "veiculo", "data_publicacao", "titulo", "resumo_fonte", "url"]].to_string())
print("Duplicatas de URL:", politica["url"].duplicated().sum())
print("Resumos vazios:", politica["resumo_fonte"].fillna("").str.strip().eq("").sum())

id                                                               P01
veiculo                                               Agência Brasil
data_publicacao                                           2026-10-12
titulo             Lula defende respeito às religiões no Dia de N...
resumo_fonte       Ao citar o Minha Casa, Minha Vida, o president...
url                https://agenciabrasil.ebc.com.br/politica/noti...
Duplicatas de URL: 0
Resumos vazios: 0


## 2. Contar menções neste recorte

Antes do modelo, conte `Lula` e `Flávio` nos títulos e resumos. O denominador é **40 matérias selecionadas**, dez por grupo. Essa contagem descreve as linhas da tabela; não estima quantas matérias os veículos publicaram ao todo. Uma notícia pode citar ambos.


In [10]:
politica["mencoes_lula"] = politica["texto_analise"].str.count(r"(?i)\bLula\b")
politica["mencoes_flavio"] = politica["texto_analise"].str.count(r"(?i)\bFlávio\b")
print(politica.groupby(["veiculo", "foco_amostral"])[["mencoes_lula", "mencoes_flavio"]].sum())


                                 mencoes_lula  mencoes_flavio
veiculo        foco_amostral                                 
Agência Brasil Flávio Bolsonaro             6              13
               Lula                        13               4
CNN Brasil     Flávio Bolsonaro             6              11
               Lula                        10               2


## 3. Preparação e tópicos: transferir o método

A base política tem textos curtos. Reaplique a limpeza da Aula 21, mas preserve nomes e negações. NMF com quatro tópicos será apenas uma lente exploratória: quarenta textos não bastam para afirmar quais são “os temas da campanha”. Leia os títulos de maior peso antes de nomear qualquer grupo.

In [11]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
politica["texto_preparado"] = politica["texto_analise"].fillna("").str.lower().map(
    lambda s: " ".join(re.findall(r"\b[^\W\d_]+\b", s, flags=re.UNICODE))
)
stopwords = "a o e de da do das dos em no na nos nas um uma para por com que se os as ao foi ser são mais também lula flávio bolsonaro".split()
import unicodedata
stopwords = [unicodedata.normalize("NFD", w).encode("ascii", "ignore").decode("ascii") for w in stopwords]
vetor = TfidfVectorizer(strip_accents="unicode", stop_words=stopwords,
                        min_df=2, max_df=0.85, max_features=1000, ngram_range=(1, 2))
X = vetor.fit_transform(politica["texto_preparado"])
nmf = NMF(n_components=4, init="nndsvda", random_state=42, max_iter=300)
pesos = nmf.fit_transform(X)
termos = vetor.get_feature_names_out()
for i, comp in enumerate(nmf.components_):
    print(i, ", ".join(termos[j] for j in comp.argsort()[-8:][::-1]))
politica["topico"] = pesos.argmax(axis=1)
print(pd.crosstab([politica.veiculo, politica.foco_amostral], politica.topico))

0 turno, segundo turno, senador, segundo, pai, candidato, presidencia, candidato presidencia
1 canal, tse, exercito, canal exercito, eleitoral, sobre, nunes, nunes marques
2 ato, rio, apoio, defesa, ato rio, reeleicao, educacao, sociais
3 campanha, casa, presidente, vida, site, oferece, site oferece, pretende
topico                           0  1  2  3
veiculo        foco_amostral               
Agência Brasil Flávio Bolsonaro  1  0  6  3
               Lula              0  3  4  3
CNN Brasil     Flávio Bolsonaro  6  2  1  1
               Lula              2  2  3  3


## 4. Sua chave Groq, criada por você

Crie uma conta e uma chave em [GroqCloud](https://console.groq.com/keys) ou use a que eu deixei aqui no link https://anotepad.com/notes/ed6bya2r . No Colab, a célula usa `getpass`: o que você digitar fica oculto. **Não cole a chave no notebook, no README nem no repositório.** Cada estudante usa sua própria chave ou usa a minha, se tiver dificuldade de criar. A demonstração faz **uma chamada**; o projeto usará uma amostra fixa de 16 chamadas adicionais. O modelo escolhido aceita saída JSON com esquema estrito; confira o uso e os limites da sua conta antes de executar.

In [ ]:
from getpass import getpass
chave_groq = getpass("Cole sua chave Groq (a entrada não aparece): ")
if not chave_groq.startswith("gsk_"):
    raise ValueError("Chave ausente ou formato inesperado. Crie sua chave no GroqCloud.")
print("Chave recebida nesta sessão; ela não será salva no notebook.")

Chave recebida nesta sessão; ela não será salva no notebook.


## 5. Extração estruturada: fato, tema e evidência

O modelo recebe **somente** o título e o resumo, nunca o artigo completo. O esquema pede pessoa central, fato descrito, tema, trecho de evidência e uma limitação. JSON válido não garante fato correto: depois da resposta, conferimos se a evidência é trecho literal do texto de análise. Se não for, o resultado não passa na auditoria.

In [13]:
import requests, json
modelo_groq = "openai/gpt-oss-20b"
esquema = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "pessoa_central": {"type": ["string", "null"]},
        "fato": {"type": "string"},
        "tema": {"type": "string"},
        "evidencia": {"type": "string"},
        "limite": {"type": "string"}
    },
    "required": ["pessoa_central", "fato", "tema", "evidencia", "limite"]
}

def extrair(texto):
    resposta = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {chave_groq}"},
        json={"model": modelo_groq, "temperature": 0, "reasoning_effort": "low", "max_completion_tokens": 600,
              "messages": [{"role": "system", "content": "Extraia apenas fatos presentes no texto recebido. A evidencia deve ser um trecho literal e contínuo do texto. Se não houver pessoa central, use null. Não infira intenção política nem viés do veículo."},
                           {"role": "user", "content": texto}],
              "response_format": {"type": "json_schema", "json_schema": {"name": "ficha_noticia", "strict": True, "schema": esquema}}},
        timeout=40)
    if resposta.status_code == 429:
        raise RuntimeError("Limite Groq atingido (429). Aguarde antes de tentar outra chamada.")
    resposta.raise_for_status()
    return json.loads(resposta.json()["choices"][0]["message"]["content"])

texto_teste = politica.loc[0, "texto_analise"]
ficha_teste = extrair(texto_teste)
print(ficha_teste)
print("Evidência literal:", ficha_teste["evidencia"] in texto_teste)

{'pessoa_central': 'Lula', 'fato': 'Lula defende respeito às religiões no Dia de Nossa Senhora Aparecida e cita o Minha Casa, Minha Vida, destacando que o tema da campanha da fraternidade da igreja Católica este ano é Fraternidade e Moradia.', 'tema': 'Religião e habitação', 'evidencia': 'Lula defende respeito às religiões no Dia de Nossa Senhora Aparecida. Ao citar o Minha Casa, Minha Vida, o presidente destacou que o tema da campanha da fraternidade da igreja Católica este ano é Fraternidade e Moradia.', 'limite': '2026-10-20'}
Evidência literal: True


**Ponto de conferência:** abra o link da matéria para verificar se o resumo representa bem o artigo. O modelo pode preencher o esquema e ainda exagerar no campo `fato`. A auditoria humana compara o texto recebido, a ficha e a página da fonte.

## 6. DSPy: declarar a tarefa como entrada e saída

DSPy é outra forma de organizar a mesma tarefa: declaramos os campos, e um programa os solicita ao modelo. Vamos ver a assinatura em código. A chamada abaixo é **opcional** e custa mais uma requisição; o Projeto 4 usa a função Groq da seção anterior para manter formato e contagem de chamadas previsíveis.

In [14]:
import dspy
class FatoNoticia(dspy.Signature):
    """Descreva somente o fato presente no texto e devolva uma evidência literal."""
    texto: str = dspy.InputField()
    fato: str = dspy.OutputField()
    evidencia: str = dspy.OutputField()
print(FatoNoticia)

FatoNoticia(texto -> fato, evidencia
    instructions='Descreva somente o fato presente no texto e devolva uma evidência literal.'
    texto = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Texto:', 'desc': '${texto}'})
    fato = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Fato:', 'desc': '${fato}'})
    evidencia = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Evidencia:', 'desc': '${evidencia}'})
)


In [15]:
# Opcional: execute apenas se quiser comparar uma chamada DSPy à ficha da seção 4.
# lm = dspy.LM("groq/openai/gpt-oss-20b", api_key=chave_groq, cache=False)
# dspy.configure(lm=lm)
# resposta_dspy = dspy.Predict(FatoNoticia)(texto=texto_teste)
# print(resposta_dspy.fato, resposta_dspy.evidencia)

## 7. Busca por palavra e por significado

A pergunta pode não usar a palavra da manchete. Vamos procurar nas **40** linhas de `texto_analise`, primeiro com TF-IDF e depois com embeddings multilíngues. Ative GPU no Colab para acelerar o segundo método. O modelo de embeddings é baixado na primeira execução; nenhuma chamada Groq é feita aqui.

In [16]:
pergunta = "Quais notícias tratam de apoio de lideranças políticas a candidatos?"
consulta_palavras = vetor.transform([pergunta.lower()])
ordem_palavras = (X @ consulta_palavras.T).toarray().ravel().argsort()[-3:][::-1]
for pos in ordem_palavras:
    linha = politica.iloc[pos]
    print(linha["id"], linha["titulo"], linha["url"])

P21 MDB em dez estados do Norte e Nordeste declaram apoio à reeleição de Lula https://www.cnnbrasil.com.br/eleicoes/mdb-em-dez-estados-do-norte-e-nordeste-declaram-apoio-a-reeleicao-de-lula/
P04 Estudantes fazem ato no Rio de Janeiro em defesa de voto em Lula https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/estudantes-fazem-ato-no-rio-de-janeiro-em-defesa-de-voto-em-lula
P37 Centrão mira pastas com maior orçamento se Flávio vencer https://www.cnnbrasil.com.br/blogs/isabel-mega/eleicoes/centrao-mira-pastas-com-maior-orcamento-se-flavio-vencer/


In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import torch
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)
modelo_busca = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=dispositivo)
vetores_noticias = modelo_busca.encode(politica["texto_analise"].tolist(), normalize_embeddings=True)
vetor_pergunta = modelo_busca.encode([pergunta], normalize_embeddings=True)
scores = cosine_similarity(vetor_pergunta, vetores_noticias)[0]
for pos in scores.argsort()[-3:][::-1]:
    linha = politica.iloc[pos]
    print(linha["id"], round(float(scores[pos]), 3), linha["titulo"], linha["url"])

W1020 18:44:56.100000 97198 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Dispositivo: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

P14 0.543 Flávio Bolsonaro recebe apoio de governadores eleitos e Cury https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/flavio-bolsonaro-recebe-apoios-de-governadores-eleitos-e-cury
P21 0.513 MDB em dez estados do Norte e Nordeste declaram apoio à reeleição de Lula https://www.cnnbrasil.com.br/eleicoes/mdb-em-dez-estados-do-norte-e-nordeste-declaram-apoio-a-reeleicao-de-lula/
P40 0.501 Michelle sinaliza preferência por exercer mandato caso Flávio seja eleito https://www.cnnbrasil.com.br/blogs/gustavo-uribe/eleicoes/michelle-sinaliza-preferencia-por-exercer-mandato-caso-flavio-seja-eleito/


**Compare os três primeiros resultados:** algum aparece só na busca semântica? Leia o título e o resumo antes de dizer que ele responde à pergunta. Recuperar um trecho relevante não prova que uma conclusão sobre candidato ou veículo seja verdadeira. Uma resposta fundamentada deve apontar IDs e links; isso é o começo de um RAG pequeno, sem esconder as fontes.

## 8. Projeto 4 — sua análise

Abra `exercicios/projeto-4-noticias.ipynb`. Ele usa a base fechada de 40 matérias para a comparação descritiva e **uma amostra balanceada de 16** para extração com Groq. Perguntas úteis: que temas surgem em cada recorte? Quem aparece como agente do fato? Em que fatos o candidato é o sujeito da ação e em quais apenas é citado? Quais buscas trazem matérias que uma palavra-chave perderia?

**Cuidado com o denominador:** escolhemos dez matérias para cada combinação de veículo e foco. Portanto, a base **não mede quantas notícias cada veículo publicou sobre cada candidato**. Ela permite comparar o conteúdo destas matérias selecionadas. Não use “tom positivo/negativo” como atalho para concluir viés sem leitura e protocolo próprio.

## Quando der errado

- **Chave ausente ou `401`:** crie sua própria chave e digite novamente; não a coloque em uma célula compartilhada.
- **`429`:** pare as chamadas e aguarde o limite da conta; não rode a célula do lote repetidamente.
- **JSON válido, evidência falsa:** marque a ficha para revisão; consulte o título, resumo e link.
- **GPU indisponível:** a busca em 40 textos funciona na CPU, apenas demora um pouco mais.
- **URL de fonte indisponível:** use os metadados congelados e registre que a checagem do artigo completo ficou pendente; não substitua linhas sem documentar.